<!-- NB_DOC_INTRO_V1 -->
# ML — Optimisation hyperparametres avec Optuna

> 📚 **Doc thematique** : [docs/03_ML.md](docs/03_ML.md) (Machine Learning classique)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Optuna** = framework d'optimisation hyperparametres moderne (TPE / CMA-ES / NSGA-II), avec **pruning** (arrete les essais mediocres tot) et **dashboards**. Ce notebook execute Optuna sur sklearn + XGBoost et montre les patterns cles : suggest, sampler, pruner, importance des hyperparametres.

## Plan

1. Setup + objectif
2. Optuna basique (LogReg + sklearn)
3. Optuna sur XGBoost (avec pruning)
4. Multi-objective (perf vs temps)
5. Importance des hyperparametres
6. Visualisations Optuna
7. Persistence (storage SQLite)
8. Pieges et anti-patterns
9. References


In [ ]:
import numpy as np
import optuna
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42
np.random.seed(SEED)
print(f"Optuna : {optuna.__version__}")

## 1. Setup — Breast Cancer Wisconsin

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
print(f"Shape : {X.shape}, classes {np.bincount(y)}")

## 2. Optuna basique — `study.optimize(objective, n_trials=...)`

In [ ]:
def objective_lr(trial):
    C = trial.suggest_float("C", 1e-4, 100, log=True)
    penalty = trial.suggest_categorical("penalty", ["l2", "l1"])
    solver = "liblinear" if penalty == "l1" else "lbfgs"
    pipe = Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(C=C, penalty=penalty, solver=solver, max_iter=2000, random_state=SEED)),
    ])
    return cross_val_score(pipe, X_tr, y_tr, cv=5, scoring="roc_auc", n_jobs=-1).mean()

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective_lr, n_trials=30, show_progress_bar=False)

print(f"Best AUC : {study.best_value:.4f}")
print(f"Best params : {study.best_params}")

## 3. Optuna avec pruning sur XGBoost-style (HistGradientBoosting)

In [ ]:
def objective_hgbm(trial):
    params = {
        "max_iter":       trial.suggest_int("max_iter", 50, 500),
        "max_depth":      trial.suggest_int("max_depth", 3, 15),
        "learning_rate":  trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 100),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-4, 10, log=True),
    }
    clf = HistGradientBoostingClassifier(**params, random_state=SEED)
    scores = cross_val_score(clf, X_tr, y_tr, cv=5, scoring="roc_auc", n_jobs=-1)
    return scores.mean()

# Avec MedianPruner : arrete les trials sous la mediane des essais a la meme epoch
study_hgbm = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(),
)
study_hgbm.optimize(objective_hgbm, n_trials=30, show_progress_bar=False)
print(f"Best HGBM AUC : {study_hgbm.best_value:.4f}")
print(f"Best params : {study_hgbm.best_params}")

## 4. Multi-objective — Pareto-optimal (perf vs temps)

In [ ]:
import time

def objective_multi(trial):
    n_est = trial.suggest_int("n_estimators", 10, 300)
    max_depth = trial.suggest_int("max_depth", 3, 20)

    clf = RandomForestClassifier(n_estimators=n_est, max_depth=max_depth, random_state=SEED, n_jobs=-1)
    t0 = time.perf_counter()
    score = cross_val_score(clf, X_tr, y_tr, cv=3, scoring="roc_auc", n_jobs=-1).mean()
    elapsed = time.perf_counter() - t0
    return score, elapsed   # maximize score, minimize time

study_mo = optuna.create_study(directions=["maximize", "minimize"], sampler=optuna.samplers.NSGAIISampler(seed=SEED))
study_mo.optimize(objective_multi, n_trials=20, show_progress_bar=False)

# Pareto front
pareto = study_mo.best_trials
print(f"Nb solutions Pareto-optimales : {len(pareto)}")
for t in pareto[:5]:
    print(f"  AUC={t.values[0]:.4f}, time={t.values[1]:.2f}s, params={t.params}")

## 5. Importance des hyperparametres

In [ ]:
importance = optuna.importance.get_param_importances(study_hgbm)
print("Importance des HP sur le score :")
for k, v in importance.items():
    print(f"  {k:25s} : {v:.3f}")

## 6. Visualisations Optuna

Utilise plotly — affiche dans le notebook ou exporte HTML.

In [ ]:
# Disponible avec : optuna.visualization (plotly) ou optuna.visualization.matplotlib

# Matplotlib version (pas besoin de plotly)
fig = optuna.visualization.matplotlib.plot_optimization_history(study_hgbm)
plt.title("Optimization history (HGBM)")
plt.show()

In [ ]:
# Importance plot
fig = optuna.visualization.matplotlib.plot_param_importances(study_hgbm)
plt.title("Param importance")
plt.show()

## 7. Persistence — reprendre une etude

```python
# Sauvegarder dans SQLite
study = optuna.create_study(
    study_name="my_xp",
    storage="sqlite:///optuna_db.sqlite",
    load_if_exists=True,   # si existe, charge ; sinon cree
    direction="maximize",
)
study.optimize(objective, n_trials=100)

# Reprendre depuis une autre session
study = optuna.load_study(study_name="my_xp", storage="sqlite:///optuna_db.sqlite")
print(study.best_value)
```

Permet de **paralleliser** sur plusieurs machines (toutes pointent vers la meme DB).


## 8. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| `n_trials=10` sur 10 params | Insuffisant. Au moins 50-100 |
| Pas de `random_state` dans le sampler | Reproductibilite cassee |
| Suggest sur des ranges trop grand | TPE galere — restreindre par intuition |
| Pas de pruner sur trials longs | Gain temps × 2-5 |
| Optuna sur le test set | TOUJOURS sur train (CV) |
| Reporter `best_value` comme perf finale | Eval sur test hold-out apres ! |
| Pas regarder l'importance des HP | Permet de simplifier la grille |


## References

### Documentation
- Optuna docs : https://optuna.readthedocs.io/
- Optuna tutorials : https://optuna.org/#code_examples
- Pruners : https://optuna.readthedocs.io/en/stable/reference/pruners.html

### Voir aussi
- [ML_Regression_Classification_CV_GridSearch.ipynb](ML_Regression_Classification_CV_GridSearch.ipynb)
- [ML_AutoML.ipynb](ML_AutoML.ipynb)
- [ML_Bagging_Boosting.ipynb](ML_Bagging_Boosting.ipynb)
